> **데이터셋 안내** — 이 노트북이 참조하는 HF 데이터셋은 공개 배포하지 않는다.
> AI Hub 원본에서 재생성하는 절차는 [docs/data/data-pipeline.md](../docs/data/data-pipeline.md)「가공 데이터셋은 배포하지 않는다 — 재현 경로」에 있다.

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
import numpy as np
import random

load_dotenv()
ROOT = Path(os.environ["DATA_ROOT"])
HF_HOME = ROOT / ".hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)

import torch
from transformers import AutoTokenizer
from datasets import load_dataset

In [ ]:
# Config
config = {
    'num_labels': 188,
    'seed': 42,
    'model_name': 'monologg/kobert',
}

In [3]:
random.seed(config['seed'])
np.random.seed(config['seed'])
torch.manual_seed(config['seed'])
torch.cuda.manual_seed_all(config['seed'])

## 데이터셋 확인

In [5]:
dataset = load_dataset("ingyoun/patent-clean-text")
dataset

README.md:   0%|          | 0.00/739 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/129M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/121M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/14.4M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/14.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/201895 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11271 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/11162 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document_id', 'invention_title', 'abstract', 'claims', 'ipc_main', 'mno', 'label_ids', 'lno'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'invention_title', 'abstract', 'claims', 'ipc_main', 'mno', 'label_ids', 'lno'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'invention_title', 'abstract', 'claims', 'ipc_main', 'mno', 'label_ids', 'lno'],
        num_rows: 11162
    })
})

## 토크나이저

In [6]:
REV = "38279184ba645e8c94d709fbe92eb5bcb47312c1"
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], trust_remote_code=True, revision=REV)

In [7]:
print(f"Vocab Size: {tokenizer.vocab_size}")
print(f"Max Length: {tokenizer.model_max_length}")
print(f"Max Length: {tokenizer.all_special_tokens}")

Vocab Size: 8002
Max Length: 512
Max Length: ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']


In [8]:
for k, v in dataset["train"][0].items():
    print(f"{k}: {v}")

document_id: kr20010002596b1
invention_title: 반도체 디바이스 시험방법 및 그의 장치
abstract: 피시험 반도체 디바이스로부터 판독되는 각 데이터와, 이들의 데이터에 동기하여 출력되는 기준 클록을 각각 약간씩 위상차가 부여된 다상 펄스의 스트로브 펄스로 샘플링하고, 이들 샘플링 출력으로부터 각 출력데이터의 변화점 위상과 기준 클록의 변화점 위상을 구하고, 이들 양자의 위상차를 각각 계측하고, 이 위상차가 미리 정한 조건의 범위내에 있는가 여부에 의하여 피시험 반도체 디바이스의 양부를 판정한다.
claims: 피시험 디바이스로부터 출력되는 기준클록의 변화점의 초기 위상위치로부터의 위상을 측정하고, 이 기준클록의 위상으로부터 이 기준클록에 동기하여 출력되는 각 데이터의 변화점까지의 위상차를 구하여, 이 위상차의 장단에 의하여 상기 피시험 디바이스의 그레이드를 평가하는 반도체 디바이스 시험방법에 있어서,각 테스트 사이클마다 초기 위상위치로부터 순차적으로 약간씩 위상차가 부여된 다상펄스를 발생시키는 단계,이 다상펄스를 스트로브펄스로 하여 각 테스트 사이클마다 상기 기준클록을 다상으로 샘플링하는 단계,이들 다상 샘플링 출력의 인접하는 출력의 비교로부터 상기 기준클록의 변화점을 검출하고, 이러한 변화점을 검출한 다상펄스의 상번호로부터 상기 기준클록의 변화점의 위상을 결정하는 단계를 포함하는 것을 특징으로 하는 반도체 디바이스 시험방법.피시험 디바이스로부터 출력되는 기준클록의 변화점의 초기 위상위치로부터의 위상을 측정하고, 이 기준클록의 위상으로부터 이 기준클록에 동기하여 출력되는 각 데이터의 변화점까지의 위상차를 구하고, 이 위상차의 장단에 의하여 상기 피시험 디바이스의 그레이드를 평가하는 반도체 디바이스 시험방법에 있어서,각 테스트 사이클마다, 출력되는 상기 기준클록의 변화점의 위상을 미리 계측하여 메모리의 그 테스트 사이클과 대응한 어드레스에 기억하여 두는 단계,상기 위상차를 구하는 것을 행하는 때에, 각 테스트 사이클마다, 

In [9]:
def build_inputs(ex):
    inputs = dict()
    fields = ["invention_title", "ipc_main", "abstract", "claims"]
    text = " ".join(str(ex[field]) for field in fields if ex[field])   # 빈 필드 skip, 개행/들여쓰기 없음
    return {
        "document_id": ex["document_id"],
        "input" : text,
        "label_ids": ex["label_ids"]
    }

In [11]:
dataset_inputs = dataset.map(build_inputs, remove_columns=dataset["train"].column_names)
type(dataset_inputs)

Map:   0%|          | 0/201895 [00:00<?, ? examples/s]

Map:   0%|          | 0/11271 [00:00<?, ? examples/s]

Map:   0%|          | 0/11162 [00:00<?, ? examples/s]

datasets.dataset_dict.DatasetDict

In [12]:
dataset_inputs

DatasetDict({
    train: Dataset({
        features: ['document_id', 'label_ids', 'input'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'label_ids', 'input'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'label_ids', 'input'],
        num_rows: 11162
    })
})

In [14]:
dataset_inputs["train"]["input"][0]

'반도체 디바이스 시험방법 및 그의 장치 G01R-031/26 피시험 반도체 디바이스로부터 판독되는 각 데이터와, 이들의 데이터에 동기하여 출력되는 기준 클록을 각각 약간씩 위상차가 부여된 다상 펄스의 스트로브 펄스로 샘플링하고, 이들 샘플링 출력으로부터 각 출력데이터의 변화점 위상과 기준 클록의 변화점 위상을 구하고, 이들 양자의 위상차를 각각 계측하고, 이 위상차가 미리 정한 조건의 범위내에 있는가 여부에 의하여 피시험 반도체 디바이스의 양부를 판정한다. 피시험 디바이스로부터 출력되는 기준클록의 변화점의 초기 위상위치로부터의 위상을 측정하고, 이 기준클록의 위상으로부터 이 기준클록에 동기하여 출력되는 각 데이터의 변화점까지의 위상차를 구하여, 이 위상차의 장단에 의하여 상기 피시험 디바이스의 그레이드를 평가하는 반도체 디바이스 시험방법에 있어서,각 테스트 사이클마다 초기 위상위치로부터 순차적으로 약간씩 위상차가 부여된 다상펄스를 발생시키는 단계,이 다상펄스를 스트로브펄스로 하여 각 테스트 사이클마다 상기 기준클록을 다상으로 샘플링하는 단계,이들 다상 샘플링 출력의 인접하는 출력의 비교로부터 상기 기준클록의 변화점을 검출하고, 이러한 변화점을 검출한 다상펄스의 상번호로부터 상기 기준클록의 변화점의 위상을 결정하는 단계를 포함하는 것을 특징으로 하는 반도체 디바이스 시험방법.피시험 디바이스로부터 출력되는 기준클록의 변화점의 초기 위상위치로부터의 위상을 측정하고, 이 기준클록의 위상으로부터 이 기준클록에 동기하여 출력되는 각 데이터의 변화점까지의 위상차를 구하고, 이 위상차의 장단에 의하여 상기 피시험 디바이스의 그레이드를 평가하는 반도체 디바이스 시험방법에 있어서,각 테스트 사이클마다, 출력되는 상기 기준클록의 변화점의 위상을 미리 계측하여 메모리의 그 테스트 사이클과 대응한 어드레스에 기억하여 두는 단계,상기 위상차를 구하는 것을 행하는 때에, 각 테스트 사이클마다, 상기 메모리의 그 테스트 사이클과 대응한 어드레스로부터 위상을 판독하여, 상기 평가를 행하기

In [17]:
def to_features(ex):
    out = tokenizer(ex["input"], truncation=True, max_length=512)
    batch_size = len(ex["input"])
    y = np.zeros((batch_size, config["num_labels"]), dtype=np.float32)
    for i, ids in enumerate(ex["label_ids"]):
        y[i, ids] = 1.0
    out["labels"] = y.tolist()
    return out

In [18]:
remove_cols = [c for c in dataset_inputs["train"].column_names if c != "document_id"]

ds_tok = dataset_inputs.map(
    to_features, 
    remove_columns=remove_cols,
    batched=True
    )

Map:   0%|          | 0/201895 [00:00<?, ? examples/s]

Map:   0%|          | 0/11271 [00:00<?, ? examples/s]

Map:   0%|          | 0/11162 [00:00<?, ? examples/s]

In [19]:
ds_tok

DatasetDict({
    train: Dataset({
        features: ['document_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11162
    })
})

In [20]:
ds_tok.push_to_hub("ingyoun/patent-clean-text-kobert-tokenized")

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/ingyoun/patent-clean-text-kobert-tokenized/commit/0dda684df88ca3acf3a7237fe67eb6d4cd64400e', commit_message='Upload dataset', commit_description='', oid='0dda684df88ca3acf3a7237fe67eb6d4cd64400e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ingyoun/patent-clean-text-kobert-tokenized', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ingyoun/patent-clean-text-kobert-tokenized'), pr_revision=None, pr_num=None)